<a href="https://colab.research.google.com/github/sanyamsh7/Agentic-RAG-Pipeline-From-Scratch/blob/main/RAG_SQL_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


What We're Building

A conversational SQL agent with RAG capabilities on top of a realistic open-source sales dataset, with a Streamlit UI. Users can ask natural language questions or request raw SQL generation. The whole thing runs in Google Colab during development, then gets deployed for org-wide access.

The Tech Stack

Data: Kaggle's "Superstore Sales" or the Northwind database (classic enterprise sales schema — orders, customers, products, regions, reps). Northwind is the better choice because it has realistic relational tables that mirror actual enterprise CRMs.

Core Components:

- LangChain — agent orchestration + SQL chain
- SQLDatabase (LangChain wrapper) — connects agent to the DB
- ChromaDB or FAISS — vector store for the RAG layer (schema docs, business glossary, past Q&A pairs)
- OpenAI GPT-4o or Google Gemini (free via API) — the LLM brain
- SQLite — lightweight DB for Colab (swappable with PostgreSQL for production)
- Streamlit — the UI layer
- ngrok — exposes Streamlit from Colab to a public URL during dev

In [ ]:
# installing dependencies
!pip install langchain langchain-community langchain-groq groq chromadb tiktoken sqlalchemy streamlit pyngrok plotly altair pandas requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.

In [ ]:
# imports and config
import os
import sqlite3
import requests
import pandas as pd
from IPython.display import display
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = ""

# quick sanity check to make sure the llm responds
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

response = llm.invoke("Reply with exactly: Groq is working.")
print(response.content)



Groq is working.


## STEP 1

In [ ]:
# download Northwind and load into SQLite
NORTHWIND_URL = "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"
db_path = "northwind.db"
print("Downloading Northwind database...")
response = requests.get(NORTHWIND_URL)

with open(db_path, "wb") as f:
    f.write(response.content)
print(f"✅ Saved to {db_path} ({os.path.getsize(db_path) / 1024:.1f} KB)")

✅ Saved to northwind.db (24124.0 KB)


In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;")
tables = [row[0] for row in cursor.fetchall()]

In [ ]:
print(f"\nTables found ({len(tables)}):")
for t in tables:
  cursor.execute(f"SELECT COUNT(*) FROM '{t}'")
  count = cursor.fetchone()[0]
  print(f"{t:<30} {count:>6} rows") # right and left padding included for formatting
conn.close()


Tables found (14):
Categories                          8 rows
CustomerCustomerDemo                0 rows
CustomerDemographics                0 rows
Customers                          93 rows
EmployeeTerritories                49 rows
Employees                           9 rows
Order Details                  609283 rows
Orders                          16282 rows
Products                           77 rows
Regions                             4 rows
Shippers                            3 rows
Suppliers                          29 rows
Territories                        53 rows
sqlite_sequence                     6 rows


In [ ]:
# deep schema exploration
conn = sqlite3.connect(db_path)

def explore_table(table_name):
  print(f"\n{'='*55}")
  print(f"TABLE: {table_name}")
  print(f"{'='*55}")
  df_info = pd.read_sql(f"PRAGMA table_info('{table_name}')", conn)
  print(df_info[["name", "type", "notnull", "pk"]].to_string(index = False))
  df_sample = pd.read_sql(f"SELECT * FROM '{table_name}' LIMIT 3", conn)
  print(f"\nSample rows:")
  display(df_sample)

In [ ]:
core_tables = ["Customers", "Orders", "Order Details", "Products",
               "Employees", "Suppliers", "Categories", "Shippers"]
for table in core_tables:
  try:
    explore_table(table)
  except Exception as e:
    print(f"Skipping {table}: {e}")
conn.close()


TABLE: Customers
        name type  notnull  pk
  CustomerID TEXT        0   1
 CompanyName TEXT        0   0
 ContactName TEXT        0   0
ContactTitle TEXT        0   0
     Address TEXT        0   0
        City TEXT        0   0
      Region TEXT        0   0
  PostalCode TEXT        0   0
     Country TEXT        0   0
       Phone TEXT        0   0
         Fax TEXT        0   0

Sample rows:


,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,Western Europe,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,Central America,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,Central America,05023,Mexico,(5) 555-3932,None



TABLE: Orders
          name     type  notnull  pk
       OrderID  INTEGER        1   1
    CustomerID     TEXT        0   0
    EmployeeID  INTEGER        0   0
     OrderDate DATETIME        0   0
  RequiredDate DATETIME        0   0
   ShippedDate DATETIME        0   0
       ShipVia  INTEGER        0   0
       Freight  NUMERIC        0   0
      ShipName     TEXT        0   0
   ShipAddress     TEXT        0   0
      ShipCity     TEXT        0   0
    ShipRegion     TEXT        0   0
ShipPostalCode     TEXT        0   0
   ShipCountry     TEXT        0   0

Sample rows:


,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
0,10248,VINET,5,2016-07-04,2016-08-01,2016-07-16,3,16.75,Vins et alcools Chevalier,59 rue de l-Abbaye,Reims,Western Europe,51100,France
1,10249,TOMSP,6,2016-07-05,2016-08-16,2016-07-10,1,22.25,Toms Spezialitäten,Luisenstr. 48,Münster,Western Europe,44087,Germany
2,10250,HANAR,4,2016-07-08,2016-08-05,2016-07-12,2,25.00,Hanari Carnes,"Rua do Paço, 67",Rio de Janeiro,South America,05454-876,Brazil



TABLE: Order Details
     name    type  notnull  pk
  OrderID INTEGER        1   1
ProductID INTEGER        1   2
UnitPrice NUMERIC        1   0
 Quantity INTEGER        1   0
 Discount    REAL        1   0

Sample rows:


...

,SupplierID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax,HomePage
0,1,Exotic Liquids,Charlotte Cooper,Purchasing Manager,49 Gilbert St.,London,British Isles,EC1 4SD,UK,(171) 555-2222,None,None
1,2,New Orleans Cajun Delights,Shelley Burke,Order Administrator,P.O. Box 78934,New Orleans,North America,70117,USA,(100) 555-4822,None,#CAJUN.HTM#
2,3,Grandma Kelly's Homestead,Regina Murphy,Sales Representative,707 Oxford Rd.,Ann Arbor,North America,48104,USA,(313) 555-5735,(313) 555-3349,None



TABLE: Categories
        name    type  notnull  pk
  CategoryID INTEGER        0   1
CategoryName    TEXT        0   0
 Description    TEXT        0   0
     Picture    BLOB        0   0

Sample rows:


,CategoryID,CategoryName,Description,Picture
0,1,Beverages,"Soft drinks, coffees, teas, beers, and ales",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00...
1,2,Condiments,"Sweet and savory sauces, relishes, spreads, an...",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00...
2,3,Confections,"Desserts, candies, and sweet breads",b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00...



TABLE: Shippers
       name    type  notnull  pk
  ShipperID INTEGER        1   1
CompanyName    TEXT        1   0
      Phone    TEXT        0   0

Sample rows:


,ShipperID,CompanyName,Phone
0,1,Speedy Express,(503) 555-9831
1,2,United Package,(503) 555-3199
2,3,Federal Shipping,(503) 555-9931


In [ ]:
# sanity check queries
# verify business queries work before the agent touches anything
conn = sqlite3.connect(db_path)
queries = {
    "Total revenue by category" : """
    SELECT c.CategoryName, ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS Revenue
    FROM [Order Details] od
    JOIN Products p ON od.ProductID = p.ProductID
    JOIN Categories c ON p.CategoryID = c.CategoryID
    GROUP BY c.CategoryName
    ORDER BY Revenue DESC;
    """,
    "Top 5 customers by order value": """
    SELECT cu.CompanyName, COUNT(DISTINCT o.OrderID) as Orders,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS TotalValue
    FROM Customers cu
    JOIN [Orders] o ON cu.CustomerID = o.CustomerID
    JOIN [Order Details] od ON o.OrderID = od.OrderID
    GROUP BY cu.CompanyName
    ORDER BY TotalValue DESC
    LIMIT 5;
    """,
    "Revenue by sales rep": """
    SELECT e.FirstName || ' ' || e.LastName AS Rep,
    COUNT(DISTINCT o.OrderID) as Deals,
    ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)),2) AS Revenue
    FROM Employees e
    JOIN Orders o ON e.EmployeeID = o.EmployeeID
    JOIN [Order Details] od ON o.OrderID = od.OrderID
    GROUP BY Rep
    ORDER BY Revenue DESC;
    """
}
for title, sql in queries.items():
  print(f"\n{'-'*55}\n{title}\n{'-'*55}")
  df = pd.read_sql(sql, conn)
  display(df)

conn.close()



-------------------------------------------------------
Total revenue by category
-------------------------------------------------------


,CategoryName,Revenue
0,Beverages,92163184.18
1,Confections,66337803.07
2,Meat/Poultry,64881147.97
3,Dairy Products,58018116.79
4,Condiments,55795126.78
5,Seafood,49921604.17
6,Produce,32701119.88
7,Grains/Cereals,28568530.34



-------------------------------------------------------
Top 5 customers by order value
-------------------------------------------------------


,CompanyName,Orders,TotalValue
0,IT,335,9745371.29
1,B's Beverages,210,6154115.34
2,Hungry Coyote Import Store,198,5698023.67
3,Rancho grande,194,5559110.08
4,Gourmet Lanchonetes,202,5552309.81



-------------------------------------------------------
Revenue by sales rep
-------------------------------------------------------


,Rep,Deals,Revenue
0,Margaret Peacock,1908,51488395.20
1,Steven Buchanan,1804,51386459.10
2,Janet Leverling,1846,50445573.76
3,Nancy Davolio,1846,49659423.23
4,Robert King,1789,49651899.30
5,Laura Callahan,1798,49281136.81
6,Michael Suyama,1754,49139966.56
7,Anne Dodsworth,1766,49019678.44
8,Andrew Fuller,1771,48314100.77


## Step 2

The problem without RAG:

When the agent gets a question, it would normally dump the entire schema into the prompt and hope the LLM figures it out. That works on tiny schemas but breaks on anything real — wrong column names, wrong table assumptions, missed joins.

What RAG solves:

Before the LLM sees the question, we retrieve only the relevant schema context — the right tables, column meanings, and business term definitions. The LLM then generates SQL against grounded, specific context instead of guessing.

What we're embedding:

- Table descriptions (what each table means in business terms)
- Column-level metadata (what Discount actually means, what units UnitPrice is in)
- A business glossary ("top customer" → order by total spend, "active" → ordered in last 12 months)
- Sample Q&A pairs (known good question → SQL mappings, few-shot examples)

In [ ]:
# write schema documents
# these are the docs that get embedded into chromadb ,
# written in plain english - the llm reads these, not the raw schema

import os
os.makedirs("schema_docs", exist_ok = True)

schema_docs = {
    "Customers.md": """
    # Table: Customers
    ## Business context
    Represents companies or individuals who have placed orders. Each customer is a business entity (B2B), not an individual consumer. CompanyName is the primary display identifier.

    ## Columns
    - CustomerID: Primary key. Used to join with the Order table via CustomerId.
    - CompanyName: The name of the customer's company. Use this in SELECT when showing customer identity.
    - ContactName: Name of the primary contact person at the company.
    - ContactTitle: Job title of the contact (e.g., Sales Manager, Owner).
    - City, Region, Country, PostalCode: Geographic location of the customer.
    - Phone, Fax: Contact numbers. Not useful for analytics.

    ## Common Use Cases
    - Filter by Country or Region for geographic analysis
    - Join with Order table on Customers.CustomerID = Orders.CustomerID
    - Use CompanyName in output, never CustomerID alone

    ## Example
    "Customers in Germany" → WHERE Country = 'Germany'
    "How many customers do we have" → SELECT COUNT(*) FROM Customers
    """,

    "Orders.md": """
    # Table: Orders (use brackets: [Orders] in SQL due to reserved word)

    ## Business Context
    Each row is a single sales order placed by a customer, assigned to a sales employee, and fulfilled by a shipper. This is the central fact table — most revenue and activity analysis flows through here.

    ## Columns
    - OrderID: Primary key. Join to OrderDetails on Orders.OrderID = OrderDetails.OrderID
    - CustomerID: Foreign key to Customers.CustomerID
    - EmployeeID: Foreign key to Employees.EmployeeID — this is the sales rep who owns the order
    - OrderDate: When the order was placed. Use for time-based analysis.
    - RequiredDate: When the customer needed it by.
    - ShippedDate: When it actually shipped. NULL means not yet shipped.
    - ShipVia: Foreign key to Shipper.Id
    - Freight: Shipping cost in USD
    - ShipCountry, ShipCity, ShipRegion: Delivery destination

    ## Important Notes
    - ALWAYS wrap in brackets: [Orders] — it is a reserved SQL keyword
    - OrderDate is the correct date field for "when was the order placed"
    - ShippedDate can be NULL for pending orders

    ## Example
    "Orders placed in 1997" → WHERE strftime('%Y', OrderDate) = '1997'
    "Unshipped orders" → WHERE ShippedDate IS NULL
    """,

    "OrderDetail.md": """
    # Table: Order Details (sometimes called Order Details with a space)

    ## Business Context
    Line items for each order. One order can have multiple products. This table holds the actual pricing and quantity data — revenue calculations always come from here.

    ## Columns
    - OrderID: Foreign key to Orders.OrderID
    - ProductID: Foreign key to Products.ProductID
    - UnitPrice: Price per unit at time of sale (may differ from current Products.UnitPrice)
    - Quantity: Number of units ordered
    - Discount: Discount applied as a decimal (0.1 = 10% off). Range: 0.0 to 1.0

    ## Revenue Formula
    Revenue = UnitPrice * Quantity * (1 - Discount)
    ALWAYS use this formula. Never use UnitPrice * Quantity alone — it ignores discounts.

    ## Example
    "Total revenue" → SUM(UnitPrice * Quantity * (1 - Discount))
    "Average discount given" → AVG(Discount) * 100 to show as percentage
    """,

    "Product.md": """
    # Table: ProductS

    ## Business Context
    The product catalog. Each product belongs to a category and comes from a supplier.

    ## Columns
    - ProductID: Primary key. Join to OrderDetail on Product.ProductID = OrderDetails.ProductID
    - ProductName: Human-readable product name. Use this in output.
    - SupplierID: Foreign key to Suppliers.SupplierID
    - CategoryID: Foreign key to Categories.CategoryID
    - QuantityPerUnit: How the product is packaged (e.g., "12 - 550 ml bottles")
    - UnitPrice: Current list price. Note: actual sale price is in OrderDetails.UnitPrice
    - UnitsInStock: Current inventory count
    - UnitsOnOrder: Units on order from supplier
    - ReorderLevel: Minimum stock before reorder
    - Discontinued: 1 if product is no longer sold, 0 if active

    ## Example
    "Active products" → WHERE Discontinued = 0
    "Low stock products" → WHERE UnitsInStock < ReorderLevel
    """,
    "Employee.md": """
    # Table: Employees

    ## Business Context
    Sales representatives and managers. When users ask about "reps", "salespeople", or "who sold the most" — this is the table.

    ## Columns
    - EmployeeID: Primary key. Join to Order on Employees.EmployeeID = Orders.EmployeeID
    - FirstName, LastName: Always concatenate for display: FirstName || ' ' || LastName AS Rep
    - Title: Job title (e.g., Sales Representative, Vice President Sales)
    - ReportsTo: Self-referencing FK to Employee.Id for org hierarchy
    - HireDate: When the employee was hired
    - Country, City: Where the employee is based

    ## Example
    "Top performing reps" → GROUP BY Employee, ORDER BY SUM(revenue) DESC
    "Sales managers" → WHERE Title LIKE '%Manager%'
    """,
      "Category.md": """
    # Table: Categories

    ## Business Context
    Product categories. Used to group products for high-level reporting.

    ## Columns
    - CategoryID: Primary key. Join to Product on Categories.CategoryID = Products.CategoryID
    - CategoryName: The display name (e.g., Beverages, Dairy Products, Seafood)
    - Description: Plain text description of the category

    ## Categories in the data
    Beverages, Condiments, Confections, Dairy Products, Grains/Cereals, Meat/Poultry, Produce, Seafood

    ## Example
    "Revenue by category" → GROUP BY Categories.CategoryName
    """,
    "Supplier.md": """
    # Table: Suppliers

    ## Business Context
    Companies that supply products. Less commonly queried directly but useful for supply chain analysis.

    ## Columns
    - SupplierID: Primary key. Join to Product on Suppliers.SupplierID = Products.SupplierID
    - CompanyName: Supplier's company name
    - Country, City: Supplier location

    ## Example
    "Products from US suppliers" → JOIN Supplier ON ... WHERE Suppliers.Country = 'USA'
    """,
        "Shipper.md": """
    # Table: Shippers

    ## Business Context
    Shipping companies used to fulfill orders.

    ## Columns
    - ShipperID: Primary key. Join to Order on Shippers.ShipperID = Orders.ShipVia
    - CompanyName: Shipper name (e.g., Federal Shipping, Speedy Express, United Package)

    ## Example
    "Most used shipper" → GROUP BY Shipper.CompanyName ORDER BY COUNT(*) DESC
    """
}

In [ ]:
for filename, content in schema_docs.items():
    with open(f"schema_docs/{filename}", "w") as f:
        f.write(content.strip())

print(f"Written {len(schema_docs)} schema documents to schema_docs/")

Written 8 schema documents to schema_docs/


In [ ]:
# Business Glossary
# maps plain english terms to their SQL equivalents
# injected into the prompt when relevant terms are detected.

import json

glossary = {
    "revenue": {
        "definition": "Total sales value after discounts",
        "sql": "SUM(od.UnitPrice * od.Quantity * (1 - od.Discount))",
        "tables_needed": ["Order Details"]
    },
    "top customer": {
        "definition": "Customer with highest total spend",
        "sql": "ORDER BY SUM(od.UnitPrice * od.Quantity * (1-od.Discount)) DESC",
        "tables_needed": ["Customers", "Orders", "Order Details"]
    },
    "active product": {
        "definition": "Product that has not been discontinued",
        "sql": "WHERE p.Discontinued = 0",
        "tables_needed": ["Products"]
    },
    "sales rep": {
        "definition": "Employee who owns orders, identified by EmployeeId on the Order table",
        "sql": "JOIN Employees e ON e.EmployeeID = o.EmployeeID",
        "tables_needed": ["Employees", "Orders"]
    },
    "deal": {
        "definition": "A single order placed by a customer",
        "sql": "COUNT(DISTINCT o.EmployeeID)",
        "tables_needed": ["Orders"]
    },
    "average order value": {
        "definition": "Mean revenue per order",
        "sql": "SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) / COUNT(DISTINCT o.EmployeeID)",
        "tables_needed": ["Orders", "Order Details"]
    },
    "discount rate": {
        "definition": "Average discount applied across orders, as a percentage",
        "sql": "ROUND(AVG(od.Discount) * 100, 2)",
        "tables_needed": ["Order Details"]
    },
    "pending orders": {
        "definition": "Orders that have been placed but not yet shipped",
        "sql": "WHERE o.ShippedDate IS NULL",
        "tables_needed": ["Orders"]
    },
    "low stock": {
        "definition": "Products where current inventory is below the reorder level",
        "sql": "WHERE p.UnitsInStock < p.ReorderLevel",
        "tables_needed": ["Products"]
    },
    "churn risk": {
        "definition": "Customers who have not placed an order in the last 12 months",
        "sql": "WHERE MAX(o.OrderDate) < DATE('now', '-12 months')",
        "tables_needed": ["Customers", "Orders"]
    }
}

with open("glossary.json", "w") as f:
    json.dump(glossary, f, indent = 2)

print(f"Glossary written with {len(glossary)} terms")
for term in glossary:
  print(f"- {term}")

Glossary written with 10 terms
- revenue
- top customer
- active product
- sales rep
- deal
- average order value
- discount rate
- pending orders
- low stock
- churn risk


In [ ]:
# Build ChromaDB vector space
# embed schema docs + glossary into ChromaDB
# this is the retrieval engine for the RAG layer

import chromadb
from chromadb.utils import embedding_functions
import glob

# use chromadb's built-in sentence transformer ( free, no api key needed )
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-v2"
)

# persistent client - survives colab session if you remount drive
client = chromadb.Client()
collection = client.get_or_create_collection(
    name = "schema_docs",
    embedding_function = emb_fn
)

documents = []
metadatas = []
ids = []

# load schema docs
for filepath in glob.glob("schema_docs/*.md"):
  with open(filepath, "r") as f:
    content = f.read()
  table_name = os.path.basename(filepath).replace(".md", "")
  documents.append(content)
  metadatas.append({"source": "schema", "table": table_name})
  ids.append(f"schema_{table_name}")

# load glossary terms as individual documents
with open("glossary.json", "r") as f:
  glossary = json.load(f)

for term, details in glossary.items():
  doc = f"Business term: {term}\nDefinition: {details['definition']}\nSQL pattern:n {details['sql']}\nTables needed: {', '.join(details['tables_needed'])}"
  documents.append(doc)
  metadatas.append({"source": "glossary", "term": term})
  ids.append(f"glossary_{term.replace(' ', '_')}")

# add everything  to chromadb
collection.add(documents=documents, metadatas = metadatas, ids=ids)

print(f"ChromaDB loaded with {collection.count()} documents")
print(f"  - Schema docs: {len(glob.glob('schema_docs/*.md'))}")
print(f"  - Glossary terms: {len(glossary)}")

In [ ]:
# test the retriever
# test the retriever before wiring it to the agent
# this is what runs every time a user asks a question
def retrieve_context(question: str, n_results: int = 3) -> str:
  """
  Given a user question, retrieve the most relevant schema
  and glossary context from chromadb.
  returns a single formatted string ready to inject into a prompt.
  """
  results = collection.query(
      query_texts = [question],
      n_results = n_results
  )

  context_parts = []
  for i , doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    source_label = f"[{meta['source'].upper()}]"
    context_parts.append(f"{source_label}\n{doc}")
  return "\n\n---\n\n".join(context_parts)

# test with real questions
test_questions = [
    "Who are the top customers by revenue?",
    "Which products are low on stock?",
    "What is the average discount rate by sales rep?",
    "Show me pending orders"
]

for q in test_questions:
  print(f"\n{'='*60}")
  print(f"QUESTION: {q}")
  print(f"{'='*60}")
  context = retrieve_context(q)
  print(context[:600] + "..." if len(context) > 600 else context)


QUESTION: Who are the top customers by revenue?
[GLOSSARY]
Business term: top customer
Definition: Customer with highest total spend
SQL pattern:n ORDER BY SUM(od.UnitPrice * od.Quantity * (1-od.Discount)) DESC
Tables needed: Customers, Orders, Order Details

---

[GLOSSARY]
Business term: revenue
Definition: Total sales value after discounts
SQL pattern:n SUM(od.UnitPrice * od.Quantity * (1 - od.Discount))
Tables needed: Order Details

---

[GLOSSARY]
Business term: average order value
Definition: Mean revenue per order
SQL pattern:n SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) / COUNT(DISTINCT o.EmployeeID)
Tables needed: Orders, O...

QUESTION: Which products are low on stock?
[GLOSSARY]
Business term: low stock
Definition: Products where current inventory is below the reorder level
SQL pattern:n WHERE p.UnitsInStock < p.ReorderLevel
Tables needed: Products

---

[SCHEMA]
# Table: ProductS

    ## Business Context
    The product catalog. Each product belongs to a category a

In [ ]:
import os, shutil, glob

# Check what's actually in schema_docs right now
print("Files in schema_docs/:")
for f in sorted(glob.glob("schema_docs/*.md")):
    print(f"\n--- {f} ---")
    with open(f) as file:
        print(file.read()[:200])

Files in schema_docs/:

--- schema_docs/Category.md ---
# Table: Categories

    ## Business Context
    Product categories. Used to group products for high-level reporting.

    ## Columns
    - CategoryID: Primary key. Join to Product on Categories.Categ

--- schema_docs/Customers.md ---
# Table: Customers
    ## Business context
    Represents companies or individuals who have placed orders. Each customer is a business entity (B2B), not an individual consumer. CompanyName is the prim

--- schema_docs/Employee.md ---
# Table: Employees

    ## Business Context
    Sales representatives and managers. When users ask about "reps", "salespeople", or "who sold the most" — this is the table.

    ## Columns
    - Employ

--- schema_docs/OrderDetail.md ---
# Table: Order Details (sometimes called Order Details with a space)

    ## Business Context
    Line items for each order. One order can have multiple products. This table holds the actual pricing a

--- schema_docs/Orders.md ---
# Tabl

Before the LLM generates any SQL, a retrieval step runs against a ChromaDB vector store that has schema documentation and a business glossary embedded using sentence transformers. So if someone asks about 'churn risk' or 'top deals', the agent already knows what those mean in SQL terms before it writes a single line of code. That's what makes it accurate on ambiguous business questions rather than just schema lookups.

# Step - 3 The SQL Agent Core

In [ ]:
# connect LangChain to the SQLite Database
# wire langchain sqldatabase to northwind sqlite

from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(
    "sqlite:///northwind.db",
    sample_rows_in_table_info = 2, # includes 2 sample rows per table in schema context
    include_tables=[                # only expose the tables we care about
        "Customers",
        "Orders",
        "Order Details",
        "Products",
        "Employees",
        "Categories",
        "Suppliers",
        "Shippers"
    ]
)

# now verify LangChain can see the schema
print("Database connected")
print(f"\nTables visible to agent : \n {db.get_usable_table_names()}")
print(f"\nSample schema info (first 1000 chars):")
print(db.get_table_info())

Database connected

Tables visible to agent : 
 ['Categories', 'Customers', 'Employees', 'Order Details', 'Orders', 'Products', 'Shippers', 'Suppliers']

Sample schema info (first 1000 chars):

CREATE TABLE "Categories" (
	"CategoryID" INTEGER, 
	"CategoryName" TEXT, 
	"Description" TEXT, 
	"Picture" BLOB, 
	PRIMARY KEY ("CategoryID")
)

/*
2 rows from Categories table:
CategoryID	CategoryName	Description	Picture
1	Beverages	Soft drinks, coffees, teas, beers, and ales	b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x0
2	Condiments	Sweet and savory sauces, relishes, spreads, and seasonings	b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x02\x00\x00d\x00d\x00\x00\xff\xec\x00\x11Ducky\x00\x01\x00\x0
*/


CREATE TABLE "Customers" (
	"CustomerID" TEXT, 
	"CompanyName" TEXT, 
	"ContactName" TEXT, 
	"ContactTitle" TEXT, 
	"Address" TEXT, 
	"City" TEXT, 
	"Region" TEXT, 
	"PostalCode" TEXT, 
	"Country" TEXT, 
	"Phone" TEXT, 
	"Fax" TEXT, 
	PRIMARY KEY (

In [ ]:
# build the customer RAG prompt
# custom prompt that injects RAG context
# this is the core of what makes this a RAG SQL AGENT
from langchain_core.prompts import PromptTemplate

RAG_SQL_PREFFIX = """You are an enterprise sales data analyst assistant.
You have access to SQLite database containing sales data with the following tables:
Customers, [Orders], [Order Details], [Products], [Employees], [Categories], [Suppliers], [Shippers].

CRITICAL SQL RULES - follow these exactly:
1. ALWAYS wrap the Orders table in brackers: [Orders] - it is a reserved SQL keyword in SQLite
2. Revenue MUST always be calculated as: SUM(UnitPrice * Quantity * (1 - Discount))
3. Never use UnitPrice * Quantity alone - always apply the Discount
4. Use table aliases: cu=Customers, o=[Orders], od=[Order Details], p=Products, e=Employees, c=Category
5. Always use human-readable names in output (CompanyName, ProductName, FirstName || ' ' || LastName)
6. Limit results to 10 rows unless the user asks for more
7. If the question is ambiguous, make a reasonable assumption and state it.
"""

print("Prompt template defined!")

Prompt template defined!


In [ ]:
# build the sql agent
# assemble the full sql agent

from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model_name = "llama-3.3-70b-versatile",
    temperature = 0,
    groq_api_key=os.environ["GROQ_API_KEY"],
    verbose = True
)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,       # shows agent's thinking - important for debugging
    agent_type="zero-shot-react-description",
    prefix=RAG_SQL_PREFFIX,
    max_iterations = 10, # prevents infinite loops
    handle_parsing_errors = True
)
print("SQL agent ready!")

SQL agent ready!


In [ ]:
# rag query functino ( context injected into question )
# query functino: rag context prepended to question

def run_rag_sql_agent(question: str, mode: str = "insight") -> dict:
  """
  Args:
    question: Natural language question
    mode: "insight" = plain englist answer
          "sql" = answer + generated sql
  """

  # step 1 : retrieve relevant schema context from ChromaDB
  rag_context = retrieve_context(question, n_results=3)

  # step 2 : prepent context directly into the question
  # this is the correct way to inject RAG context with create_sql_agent
  enriched_question = f"""Use the following schema context to help answer the question accurately.

  SCHEMA CONTEXT:
  {rag_context}

  QUESTION:
  {question}"""

  # step 3 : run the agent
  try:
    response = agent.invoke({"input": enriched_question})
    answer = response.get("output", "No answer returned.")

    if mode == "sql":
      sql_query = None
      steps = response.get("intermediate_steps", [])
      for step in steps:
        if isinstance(step, tuple) and len(step) == 2:
          action = step[0]
          if hasattr(action, "tool") and action.tool == "sql_db_query":
            sql_query = action.tool_input
            break
      return {"answer": answer, "sql": sql_query or "SQL not captured", "error": None}

    return {"answer": answer, "sql":None, "error":None}

  except Exception as e:
    return {"answer": None, "sql": None, "error": str(e)}

print("run_rag_sql_agent defined!")

run_rag_sql_agent defined!


In [ ]:
# # test the agent end to end
# # test both modes with real questions
# # read the verbose output - it shows the agent's reasoning chain

# test_cases = [
#     {
#         "question": "Who are the top 5 customers by total revenue?",
#         "mode": "insight"
#     },
#     {
#         "question": "Which product categories generate the most revenue?",
#         "mode": "insight"
#     },
#     {
#         "question": "Show me all pending orders",
#         "mode": "sql"
#     },
#     {
#         "question": "Which sales rep closed the most deals?",
#         "mode": "sql"
#     }
# ]

# for test in test_cases:
#   print(f"\n{'='*60}")
#   print(f"MODE    : {test['mode'].upper()}")
#   print(f"QUESTION: {test['question']}")
#   print(f"{'='*60}")
#   result = run_rag_sql_agent(test["question"], mode=test["mode"])
#   if result["error"]:
#     print(f"ERROR   : {result['error']}")
#   else:
#     print(f"ANSWER  : {result['answer']}")
#     if result["sql"]:
#       print(f"\n GENERATED SQL:\n{result['sql']}")

The agent runs a ReAct loop — Reason, Act, Observe, repeat. On each iteration it decides which tool to use, either inspecting the schema or executing SQL. Before any of that starts, the RAG context is already injected into the system prompt so the LLM knows which tables are relevant and what the business terms mean. The result is that it rarely hallucinates column names because it's working from grounded documentation rather than the raw schema alone.

# step -4

In [ ]:
# install plotly
try:
  import plotly.express as px
  print("already installed")
except ImportError:
  !pip install plotly
  import plotly.express as px
  print("installed")

already installed


In [ ]:
# result parser
# parse agent output into structured result
# extract tabular data from the agent's answer if present

import re
import pandas as pd

def parse_agent_output(answer: str) -> dict:
  """
  Takes the raw agent answer string and tries to extract structured tabular data from it .

  returns:
    dict with keys:
      - text: the clean natural language answer
      - dataframe: a pandas DataFrame if tabular data found, else None
      - has_table: bool
  """
  # look for patterns like "1. Company: X, Revenue: Y" or markdown tables
  lines = answer.strip().split("\n")

  # attempt to detect markdown table (| col | col |)
  table_lines = [l for l in lines if l.strip().startswith("|")]
  if len(table_lines) >= 3:
    try:
      # strip leading/trailing pipes and split
      headers = [h.strip() for h in table_lines[0].split("|") if h.strip()]
      rows = []
      for row_line in table_lines[2:]: # skipping header separator line
        cells = [c.strip() for c in row_line.split("|") if c.strip()]
        if len(cells) == len(headers):
          rows.append(cells)
      if rows:
        df = pd.DataFrame(rows, columns=headers)
        return {"text": answer, "dataframe": df, "has_table": True}
    except Exception :
      pass

  # attempt to detect numbered list with key:value pairs
  # e.g. "1. QUICK-Stop: $110,277"
  list_pattern = re.compile(r"^\d+[\.\)]\s+(.+?):\s+(.+)$")
  list_rows = []
  list_headers = None
  for line in lines:
    match = list_pattern.match(line.strip())
    if match:
      list_rows.append([match.group(1).strip(), match.group(2).strip()])

  if len(list_rows) >= 2:
    # trying to infer headers from context
    list_headers = ["Name", "Value"]
    df = pd.DataFrame(list_rows, columns=list_headers)
    return {"text": answer, "dataframe": df, "has_table": True}

  # no tabular data found
  return {"text": answer, "dataframe": None, "has_table": False}

print("parse_agent_output defined!")




parse_agent_output defined!


In [ ]:
# chart generator
# auto chart logic
# decides chart type based on the data shape

import plotly.express as px

def generate_chart(df: pd.DataFrame, question: str):
  """
  Given a data frame and the original question,
  pick the best chart type and return a plotly figure,
  returns: None if chart can't be determined.
  """
  if df is None or df.empty:
        return None

  cols = list(df.columns)
  question_lower = question.lower()

  # need at least 2 columns to chart
  if len(cols) < 2:
      return None

  label_col = cols[0]   # first col is usually the category/name
  value_col = cols[1]   # second col is usually the metric

  # try to coerce value column to numeric
  try:
      df = df.copy()
      df[value_col] = df[value_col].str.replace(
          r"[,$%]", "", regex=True
      ).astype(float)
  except Exception:
      return None  # can't chart non-numeric values

  # pick chart type based on question keywords
  time_keywords = ["month", "year", "quarter", "weekly", "daily", "trend", "over time"]
  bar_keywords = ["top", "most", "highest", "lowest", "compare", "by", "ranking"]

  if any(kw in question_lower for kw in time_keywords):
      fig = px.line(
          df, x=label_col, y=value_col,
          title=question,
          markers=True
      )
  elif any(kw in question_lower for kw in bar_keywords):
      fig = px.bar(
          df, x=value_col, y=label_col,
          orientation="h",
          title=question,
          color=value_col,
          color_continuous_scale="Blues"
      )
  else:
      # default to bar
      fig = px.bar(
          df, x=label_col, y=value_col,
          title=question
      )

  fig.update_layout(
      plot_bgcolor="#1e1e1e",
      paper_bgcolor="#1e1e1e",
      font_color="#ffffff",
      title_font_size=14,
      showlegend=False,
      margin=dict(l=20, r=20, t=40, b=20)
  )

  return fig


print("generate_chart() defined")

generate_chart() defined


In [ ]:
# conversation history manager
# keeps running log of questions + answers for the chat ui

from datetime import datetime

conversation_history = []

def add_to_history(question: str, result: dict, mode: str):
  """ add a complete query to the conversation log"""
  conversation_history.append({
      "timestamp": datetime.now().strftime("%H:%M:%S"),
      "question": question,
      "answer": result.get("answer"),
      "sql": result.get("sql"),
      "mode": mode,
      "error": result.get("error")
  })

def get_history() -> list:
  return conversation_history

def clear_history():
  conversation_history.clear()
  print("Conversatino history cleared.")

print("conversation_history manager defined!")

conversation_history manager defined!


In [ ]:
# master query function (combines everything)
# master functino that ties all of step-4 together
# this is what the streamlit ui will call

def query(question: str, mode: str = "insight") -> dict:
  """
  Full Pipeline:
  question -> RAG retrieval -> sql agent -> parse -> chart -> history
  args:
    question: user's natural language question
    mode: "insight" or "sql"

  returns:
    dict with keys:
      - answer: str
      - sql: str or None
      - dataframe: DataFrame or None
      - chart: Plotly figure or None
      - mode: str
      - error: str or None
  """

  print(f"\n[{mode.upper()} MODE] {question}")

  # run the agent
  result = run_rag_sql_agent(question, mode =mode)

  if result["error"]:
    print(f"❌ {result['error']}")
    add_to_history(question, result, mode)
    return {**result, "dataframe": None, "chart": None, "mode": mode}

  # parse the output for tabular data
  parsed = parse_agent_output(result["answer"])

  # generate chart if insight mode and tabular data found
  chart = None
  if mode == "insight" and parsed["has_table"]:
    chart = generate_chart(parsed["dataframe"], question)

  final_result = {
      "answer": parsed["text"],
      "sql": result["sql"],
      "dataframe": parsed["dataframe"],
      "chart": chart,
      "mode":mode,
      "error": None
  }

  add_to_history(question, result, mode)
  return final_result

print("query() defined!")

query() defined!


In [ ]:
# test the full pipeline
# end to end test of dual mode pipeline
test_cases = [
    ("What are the top 5 customers by total revenue?", "insight"),
    ("Which product categories generate the most revenue?", "insight"),
    ("Show me all pending orders with customer names", "sql"),
    ("Which sales rep closed the most deals?", "sql")
]

for question, mode in test_cases:
  print(f"\n{'='*60}")
  print(f"MODE    : {mode.upper()}")
  print(f"QUESTION: {question}")
  print(f"{'='*60}")

  result = query(question, mode=mode)

  if result["error"]:
      print(f"❌ ERROR: {result['error']}")
      continue

  print(f"\n💬 ANSWER:\n{result['answer']}")

  if result["sql"]:
      print(f"\n🔍 SQL:\n{result['sql']}")

  if result["dataframe"] is not None:
      print(f"\n📊 PARSED TABLE:")
      print(result["dataframe"].to_string(index=False))
      print(f"   Chart generated: {'✅' if result['chart'] else '❌ (non-numeric or no match)'}")

# show conversation history
print(f"\n\n{'='*60}")
print(f"CONVERSATION HISTORY ({len(get_history())} queries logged)")
print(f"{'='*60}")
for i, entry in enumerate(get_history(), 1):
    status = "✅" if not entry["error"] else "❌"
    print(f"{i}. [{entry['timestamp']}] {status} [{entry['mode'].upper()}] {entry['question'][:60]}")



MODE    : INSIGHT
QUESTION: What are the top 5 customers by total revenue?

[INSIGHT MODE] What are the top 5 customers by total revenue?


> Entering new SQL Agent Executor chain...
Thought: I should look at the tables in the database to see what I can query.  Then I should query the schema of the most relevant tables.
Action: sql_db_list_tables
Action Input: Categories, Customers, Employees, Order Details, Orders, Products, Shippers, SuppliersThought: I have the list of tables in the database. The tables that are relevant to the question are Customers, [Orders], and [Order Details]. I should now query the schema of these tables to understand their structure.

Action: sql_db_schema
Action Input: Customers, [Orders], [Order Details]Error: table_names {'[Order Details]', '[Orders]'} not found in databaseIt seems like the table names I provided earlier were not correct. The error message indicates that the table names '[Order Details]' and '[Orders]' were not found in the database. Howe

The agent output goes through a post-processing layer that detects whether the answer contains tabular data — either a markdown table or a numbered list. If it does and we're in insight mode, we auto-generate a Plotly chart. The chart type is selected based on keywords in the original question — time-related words get a line chart, ranking or comparison words get a horizontal bar. SQL mode bypasses charting entirely and returns the raw query alongside the answer. Everything gets logged to a conversation history object that the UI reads to render the chat thread.

In [ ]:
# write the streamlit app to a file
# write app.py to a disk

app_code = '''
import streamlit as st
import sys
import os
sys.path.insert(0, os.path.dirname(__file__))

# ── page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="Sales Intelligence Agent",
    page_icon="📊",
    layout="wide"
)

# ── imports ──────────────────────────────────────────────────
import json
import sqlite3
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from datetime import datetime
import plotly.express as px
import re

# ── config ───────────────────────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
DB_PATH      = "northwind.db"
CHROMA_PATH  = "schema_docs"
GLOSSARY     = "glossary.json"
MODEL        = "llama-3.3-70b-versatile"

# ── cached resource setup ────────────────────────────────────
@st.cache_resource
def load_agent():
    llm = ChatGroq(model=MODEL, temperature=0, groq_api_key=GROQ_API_KEY)
    db  = SQLDatabase.from_uri(
        f"sqlite:///{DB_PATH}",
        sample_rows_in_table_info=2,
        include_tables=["Customers","Orders","Order Details",
                        "Products","Employees","Categories",
                        "Suppliers","Shippers"]
    )
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)

    prefix = """You are an enterprise sales data analyst assistant.
You have access to a SQLite Northwind sales database.
Tables: Customers, Orders, Order Details, Products, Employees, Categories, Suppliers, Shippers.

RULES:
1. Revenue = SUM(UnitPrice * Quantity * (1 - Discount)) — always apply discount
2. Use human-readable names in output (CompanyName, ProductName, FirstName || \' \' || LastName)
3. Default LIMIT 10 unless user asks for more
4. For rep names: FirstName || \' \' || LastName AS Rep"""

    agent = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        agent_type="zero-shot-react-description",
        prefix=prefix,
        verbose=False,
        max_iterations=6,
        handle_parsing_errors=True
    )
    return agent

@st.cache_resource
def load_retriever():
    emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    client     = chromadb.Client()
    collection = client.get_or_create_collection(
        name="schema_rag_ui", embedding_function=emb_fn
    )

    import glob
    docs, metas, ids = [], [], []

    for filepath in glob.glob("schema_docs/*.md"):
        with open(filepath) as f:
            content = f.read()
        table = os.path.basename(filepath).replace(".md","")
        docs.append(content)
        metas.append({"source":"schema","table":table})
        ids.append(f"schema_{table}")

    with open(GLOSSARY) as f:
        glossary = json.load(f)
    for term, details in glossary.items():
        doc = f"Term: {term}\\nDefinition: {details[\'definition\']}\\nSQL: {details[\'sql\']}\\nTables: {\', \'.join(details[\'tables_needed\'])}"
        docs.append(doc)
        metas.append({"source":"glossary","term":term})
        ids.append(f"glossary_{term.replace(\' \',\'_\')}")

    if docs:
        collection.add(documents=docs, metadatas=metas, ids=ids)
    return collection

# ── helper functions ─────────────────────────────────────────
def retrieve_context(collection, question, n=3):
    results = collection.query(query_texts=[question], n_results=n)
    parts   = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        parts.append(f"[{meta[\'source\'].upper()}]\\n{doc}")
    return "\\n\\n---\\n\\n".join(parts)

def parse_output(answer):
    lines       = answer.strip().split("\\n")
    table_lines = [l for l in lines if l.strip().startswith("|")]
    if len(table_lines) >= 3:
        try:
            headers = [h.strip() for h in table_lines[0].split("|") if h.strip()]
            rows    = []
            for row_line in table_lines[2:]:
                cells = [c.strip() for c in row_line.split("|") if c.strip()]
                if len(cells) == len(headers):
                    rows.append(cells)
            if rows:
                return pd.DataFrame(rows, columns=headers)
        except Exception:
            pass

    list_pattern = re.compile(r"^\\d+[\\.)\\s]+(.+?)[:\\-]\\s+(.+)$")
    rows = []
    for line in lines:
        m = list_pattern.match(line.strip())
        if m:
            rows.append([m.group(1).strip(), m.group(2).strip()])
    if len(rows) >= 2:
        return pd.DataFrame(rows, columns=["Name","Value"])
    return None

def make_chart(df, question):
    if df is None or df.empty or len(df.columns) < 2:
        return None
    label_col, value_col = df.columns[0], df.columns[1]
    try:
        df        = df.copy()
        df[value_col] = df[value_col].str.replace(r"[,$%]","",regex=True).astype(float)
    except Exception:
        return None
    q = question.lower()
    if any(k in q for k in ["month","year","quarter","trend","over time"]):
        fig = px.line(df, x=label_col, y=value_col, title=question, markers=True)
    else:
        fig = px.bar(df, x=value_col, y=label_col, orientation="h",
                     title=question, color=value_col,
                     color_continuous_scale="Blues")
    fig.update_layout(
        plot_bgcolor="#0e1117", paper_bgcolor="#0e1117",
        font_color="#ffffff", showlegend=False,
        margin=dict(l=20,r=20,t=40,b=20)
    )
    return fig

def run_query(agent, collection, question, mode):
    context  = retrieve_context(collection, question)
    enriched = f"SCHEMA CONTEXT:\\n{context}\\n\\nQUESTION: {question}"
    try:
        response = agent.invoke({"input": enriched})
        answer   = response.get("output","No answer returned.")

        sql = None
        if mode == "sql":
            for step in response.get("intermediate_steps",[]):
                if isinstance(step, tuple) and len(step)==2:
                    action    = step[0]
                    tool_name = getattr(action,"tool",None) or getattr(action,"name",None)
                    tool_input= getattr(action,"tool_input",None)
                    if tool_name == "sql_db_query" and tool_input:
                        sql = tool_input
                        break
                    if not sql and tool_name == "sql_db_query_checker" and tool_input:
                        sql = tool_input

        df    = parse_output(answer) if mode == "insight" else None
        chart = make_chart(df, question) if df is not None else None
        return {"answer":answer, "sql":sql, "dataframe":df, "chart":chart, "error":None}
    except Exception as e:
        return {"answer":None,"sql":None,"dataframe":None,"chart":None,"error":str(e)}

# ── session state ─────────────────────────────────────────────
if "history" not in st.session_state:
    st.session_state.history = []
if "mode" not in st.session_state:
    st.session_state.mode = "insight"

# ── load resources ────────────────────────────────────────────
agent      = load_agent()
collection = load_retriever()

# ── sidebar ───────────────────────────────────────────────────
with st.sidebar:
    st.title("📊 Sales Agent")
    st.markdown("---")

    st.subheader("Query Mode")
    mode = st.radio(
        label="mode",
        options=["insight","sql"],
        format_func=lambda x: "💬 Insight" if x=="insight" else "🔍 SQL",
        index=0,
        label_visibility="collapsed"
    )
    st.session_state.mode = mode

    st.markdown("---")
    st.subheader("Example Questions")
    examples = [
        "Top 5 customers by revenue",
        "Revenue by product category",
        "Which sales rep closed the most deals?",
        "Show pending orders with customer names",
        "Which products are low on stock?",
        "Average discount rate by sales rep",
    ]
    for ex in examples:
        if st.button(ex, use_container_width=True):
            st.session_state.prefill = ex

    st.markdown("---")
    st.subheader("Schema Explorer")
    conn   = sqlite3.connect(DB_PATH)
    tables = ["Customers","Orders","Order Details","Products",
              "Employees","Categories","Suppliers","Shippers"]
    sel    = st.selectbox("View table", tables)
    if sel:
        df_schema = pd.read_sql(
            f\'SELECT * FROM "{sel}" LIMIT 5\', conn
        )
        st.dataframe(df_schema, use_container_width=True)
    conn.close()

    st.markdown("---")
    if st.button("🗑 Clear chat", use_container_width=True):
        st.session_state.history = []
        st.rerun()

# ── main area ─────────────────────────────────────────────────
st.title("Sales Intelligence Agent")
st.caption("Ask questions about sales data in plain English — or switch to SQL mode to see the generated query.")
st.markdown("---")

# chat history
for entry in st.session_state.history:
    with st.chat_message("user"):
        st.write(entry["question"])
    with st.chat_message("assistant"):
        st.write(entry["answer"])
        if entry.get("chart") is not None:
            st.plotly_chart(entry["chart"], use_container_width=True)
        if entry.get("sql"):
            st.code(entry["sql"], language="sql")
        if entry.get("error"):
            st.error(entry["error"])

# input box
prefill = st.session_state.pop("prefill", "")
question = st.chat_input("Ask a question about your sales data...")

if not question and prefill:
    question = prefill

if question:
    with st.chat_message("user"):
        st.write(question)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = run_query(
                agent, collection, question, st.session_state.mode
            )

        if result["error"]:
            st.error(result["error"])
        else:
            st.write(result["answer"])
            if result["chart"] is not None:
                st.plotly_chart(result["chart"], use_container_width=True)
            if result["sql"]:
                st.subheader("Generated SQL")
                st.code(result["sql"], language="sql")

    st.session_state.history.append({
        "question" : question,
        "answer"   : result.get("answer",""),
        "sql"      : result.get("sql"),
        "chart"    : result.get("chart"),
        "error"    : result.get("error")
    })
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written to disk")

✅ app.py written to disk


In [ ]:
# Launch the streamlit app with ngrok
# launch streamlit + expose via ngrok
# get your free authtoken at dashboard.ngrok.com
!pip install pyngrok -q

from pyngrok import ngrok, conf
import subprocess, time, os

NGROK_TOKEN = "3AzRpVDydAYl9tHrcIYa9LsAmIt_56eVc4RAZGSaZWY9d7sEc"
conf.get_default().auth_token = NGROK_TOKEN

# kill any existing streamlit or ngrok processes
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(2)

# start streamlit in background
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false" ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

time.sleep(4)

# open ngrok tunnel
public_url = ngrok.connect(addr="8501")
print(f"✅ App is live at: {public_url}")

✅ App is live at: NgrokTunnel: "https://supermodest-intercolonially-dacia.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
# keep alive
# check tunnel status / restart if dropped
tunnels = ngrok.get_tunnels()
if tunnels:
  print("Tunnel active: ", tunnels[0].public_url)
else:
  print("Tunnel dropped: restarting... ")
  public_url = ngrok.connect(addr="8501")
  print(f"✅ App is live at: {public_url}")

Tunnel active:  https://supermodest-intercolonially-dacia.ngrok-free.dev


In [ ]:
# # to disconnect ui app
# import os
# from pyngrok import ngrok

# ngrok.disconnect(public_url)
# os.system("pkill -f streamlit")
# os.system("pkill -f ngrok")
# print("streamlit and ngrok stopped")

streamlit and ngrok stopped


# Step 6 : Deployment

In [ ]:
# # package everything for github

# import shutil, os, zipfile

# #export folder
# os.makedirs("export/schema_docs", exist_ok=True)
# os.makedirs("export/.streamlit", exist_ok= True)
